In [ ]:
import threading # Import the threading module for concurrent execution
import sys # Import the sys module to access system-specific parameters and functions
import time # Import the time module for time-related functions
import pyaudio # Import PyAudio for audio input/output
import numpy as np # Import NumPy for numerical operations, especially with arrays
import matplotlib.pyplot as plt # Import Matplotlib for plotting waveforms
import wave # Import the wave module to work with WAV audio files
import speech_recognition as sr # Import the SpeechRecognition library
from speech_recognition import AudioData # Import AudioData class for handling audio data

stop_event = threading.Event() # Create a threading.Event object to signal when to stop recording

def wait_for_enter(): # Define a function to wait for the user to press Enter
    input("\n🎤 Press Enter to stop recording...\n") # Prompt user to press Enter
    stop_event.set() # Set the stop_event to signal recording to stop

def spinner(): # Define a function to display a spinning indicator during recording
    chars = '|/-\\' # Characters for the spinner animation (fixed: escaped backslash)
    i = 0 # Initialize counter
    while not stop_event.is_set(): # Loop while the stop_event is not set
        sys.stdout.write(f'\r🔴 Recording... {chars[i % 4]}') # Print spinner animation
        sys.stdout.flush() # Flush stdout to update the console immediately
        i += 1 # Increment counter
        time.sleep(0.1) # Pause for a short duration
    print("\r✅ Recording complete!          ") # Print recording complete message

def record_audio(): # Define a function to record audio from the microphone
    p = pyaudio.PyAudio() # Initialize PyAudio
    stream = p.open(format=pyaudio.paInt16, channels=1, rate=16000,
                    input=True, frames_per_buffer=1024) # Open an audio stream
    frames = [] # List to store audio frames

    threading.Thread(target=wait_for_enter, daemon=True).start() # Start a thread to wait for Enter key press
    threading.Thread(target=spinner, daemon=True).start() # Start a thread to display the spinner

    while not stop_event.is_set(): # Loop while the stop_event is not set (i.e., recording)
        frames.append(stream.read(1024)) # Read audio data from the stream

    stream.stop_stream() # Stop the audio stream
    stream.close() # Close the audio stream
    width = p.get_sample_size(pyaudio.paInt16) # Get the sample width
    p.terminate() # Terminate PyAudio
    return b''.join(frames), 16000, width # Return audio data, sample rate, and sample width

def save_audio(data, rate, width, filename="recording.wav"): # Define a function to save audio to a WAV file
    with wave.open(filename, 'wb') as wf: # Open a WAV file in write binary mode
        wf.setnchannels(1) # Set number of channels to 1 (mono)
        wf.setsampwidth(width) # Set sample width
        wf.setframerate(rate) # Set frame rate
        wf.writeframes(data) # Write audio data to the file
    print(f"💾 Saved: {filename}") # Print a confirmation message

def transcribe(data, rate, width): # Define a function to transcribe audio to text
    recognizer = sr.Recognizer() # Initialize the recognizer
    audio = AudioData(data, rate, width) # Create an AudioData object from raw audio
    try: # Try to recognize the speech
        text = recognizer.recognize_google(audio) # Use Google Web Speech API for transcription
        print(f"📝 Transcription: {text}") # Print the transcribed text
    except sr.UnknownValueError: # Catch error if speech is unintelligible
        print("❌ Could not understand audio") # Print error message
    except sr.RequestError as e: # Catch error if there's an issue with the API request
        print(f"❌ API Error: {e}") # Print API error message

def plot_waveform(data, rate): # Define a function to plot the audio waveform
    samples = np.frombuffer(data, dtype=np.int16) # Convert audio data to a NumPy array of 16-bit integers
    time_axis = np.linspace(0, len(samples) / rate, len(samples)) # Create a time axis for plotting
    plt.figure(figsize=(10, 4)) # Create a new figure with a specified size
    plt.plot(time_axis, samples, color='blue') # Plot the waveform
    plt.title("Your Voice Waveform") # Set plot title
    plt.xlabel("Time (seconds)") # Set x-axis label
    plt.ylabel("Amplitude") # Set y-axis label
    plt.grid(True, alpha=0.3) # Add a grid to the plot
    plt.tight_layout() # Adjust plot to prevent labels from overlapping
    plt.show() # Display the plot

def main(): # Define the main function to orchestrate the process
    print("=" * 40) # Print a separator line
    print("🎙️  HELLO AI, CAN YOU HEAR ME?") # Print a greeting message
    print("=" * 40) # Print another separator line
    print("\nSpeak into your microphone...") # Instruct user to speak

    audio_data, rate, width = record_audio() # Record audio
    save_audio(audio_data, rate, width) # Save the recorded audio
    transcribe(audio_data, rate, width) # Transcribe the recorded audio
    plot_waveform(audio_data, rate) # Plot the waveform of the recorded audio

if __name__ == "__main__": # Ensure main() runs only when the script is executed directly
    main() # Call the main function


🎙️  HELLO AI, CAN YOU HEAR ME?

Speak into your microphone...


OSError: [Errno -9996] Invalid input device (no default output device)

In [ ]:
"""

L1: Basic Speech-to-Text

Record → Save → Transcribe → Visualize Waveform



============== DEPENDENCY SETUP ==============

CHECK IF INSTALLED: pip show SpeechRecognition pyaudio numpy matplotlib



INSTALL - WINDOWS:

pip install SpeechRecognition pyaudio numpy matplotlib



INSTALL - macOS:

brew install portaudio

pip install SpeechRecognition pyaudio numpy matplotlib

==============================================

"""



"""
L1: Basic Speech-to-Text
Record → Save → Transcribe → Visualize Waveform
"""
import threading
import sys
import time
import pyaudio
import numpy as np
import matplotlib.pyplot as plt
import wave
import speech_recognition as sr
from speech_recognition import AudioData

stop_event = threading.Event()

def wait_for_enter():
    input("\n🎤 Press Enter to stop recording...\n")
    stop_event.set()

def spinner():
    chars = '|/-\\'
    i = 0
    while not stop_event.is_set():
        sys.stdout.write(f'\r🔴 Recording... {chars[i % 4]}')
        sys.stdout.flush()
        i += 1
        time.sleep(0.1)
    print("\r✅ Recording complete!          ")

def record_audio():
    p = pyaudio.PyAudio()
    stream = p.open(format=pyaudio.paInt16, channels=1, rate=16000,
                    input=True, frames_per_buffer=1024)
    frames = []

    threading.Thread(target=wait_for_enter, daemon=True).start()
    threading.Thread(target=spinner, daemon=True).start()

    while not stop_event.is_set():
        frames.append(stream.read(1024))

    stream.stop_stream()
    stream.close()
    width = p.get_sample_size(pyaudio.paInt16)
    p.terminate()
    return b''.join(frames), 16000, width

def save_audio(data, rate, width, filename="recording.wav"):
    with wave.open(filename, 'wb') as wf:
        wf.setnchannels(1)
        wf.setsampwidth(width)
        wf.setframerate(rate)
        wf.writeframes(data)
    print(f"💾 Saved: {filename}")

def transcribe(data, rate, width):
    recognizer = sr.Recognizer()
    audio = AudioData(data, rate, width)
    try:
        text = recognizer.recognize_google(audio)
        print(f"📝 Transcription: {text}")
    except sr.UnknownValueError:
        print("❌ Could not understand audio")
    except sr.RequestError as e:
        print(f"❌ API Error: {e}")

def plot_waveform(data, rate):
    samples = np.frombuffer(data, dtype=np.int16)
    time_axis = np.linspace(0, len(samples) / rate, len(samples))
    plt.figure(figsize=(10, 4))
    plt.plot(time_axis, samples, color='blue')
    plt.title("Your Voice Waveform")
    plt.xlabel("Time (seconds)")
    plt.ylabel("Amplitude")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

def main():
    print("=" * 40)
    print("🎙️  HELLO AI, CAN YOU HEAR ME?")
    print("=" * 40)
    print("\nSpeak into your microphone...")

    audio_data, rate, width = record_audio()
    save_audio(audio_data, rate, width)
    transcribe(audio_data, rate, width)
    plot_waveform(audio_data, rate)

if __name__ == "__main__":
    main()

To ensure all dependencies are installed, you can run the following in your terminal or a new code cell:


In [ ]:
!pip install pyaudio numpy matplotlib SpeechRecognition

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.1/47.1 kB 2.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.9/32.9 MB 36.7 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × Building wheel for pyaudio (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for pyaudio
Failed to build pyaudio
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (pyaudio)


In [ ]:
# Install PortAudio development libraries for PyAudio
!apt-get install -y portaudio19-dev
!pip install pyaudio numpy matplotlib SpeechRecognition

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
portaudio19-dev is already the newest version (19.6.0-1.1).
0 upgraded, 0 newly installed, 0 to remove and 7 not upgraded.


In [ ]:
# Install required package
!pip install SpeechRecognition

import io
import wave
import numpy as np
import matplotlib.pyplot as plt
import speech_recognition as sr
from IPython.display import Javascript, display, Audio
from google.colab import output
from base64 import b64decode
from speech_recognition import AudioData

def record_audio_colab(seconds=5, filename="recording.wav"):
    js = f"""
    async function recordAudio() {{
      const stream = await navigator.mediaDevices.getUserMedia({{audio:true}});
      const recorder = new MediaRecorder(stream);
      let chunks = [];

      recorder.ondataavailable = e => chunks.push(e.data);
      recorder.start();

      await new Promise(resolve => setTimeout(resolve, {seconds * 1000}));

      recorder.stop();
      await new Promise(resolve => recorder.onstop = resolve);

      const blob = new Blob(chunks, {{ type: 'audio/webm' }});
      const arrayBuffer = await blob.arrayBuffer();
      const bytes = new Uint8Array(arrayBuffer);

      let binary = '';
      bytes.forEach(b => binary += String.fromCharCode(b));
      return btoa(binary);
    }}
    recordAudio();
    """

    print(f"🎤 Recording for {seconds} seconds...")
    audio_base64 = output.eval_js(js)
    audio_bytes = b64decode(audio_base64)

    with open("temp_audio.webm", "wb") as f:
        f.write(audio_bytes)

    return "temp_audio.webm"

def convert_webm_to_wav(webm_file, wav_file="recording.wav"):
    # Colab usually has ffmpeg available
    !ffmpeg -i "$webm_file" -ac 1 -ar 16000 -y "$wav_file" > /dev/null 2>&1
    return wav_file

def read_wav_bytes(filename):
    with wave.open(filename, 'rb') as wf:
        rate = wf.getframerate()
        width = wf.getsampwidth()
        frames = wf.readframes(wf.getnframes())
    return frames, rate, width

def save_audio_info(filename):
    print(f"💾 Saved: {filename}")

def transcribe(data, rate, width):
    recognizer = sr.Recognizer()
    audio = AudioData(data, rate, width)
    try:
        text = recognizer.recognize_google(audio)
        print(f"📝 Transcription: {text}")
    except sr.UnknownValueError:
        print("❌ Could not understand audio")
    except sr.RequestError as e:
        print(f"❌ API Error: {e}")

def plot_waveform(data, rate):
    samples = np.frombuffer(data, dtype=np.int16)
    time_axis = np.linspace(0, len(samples) / rate, len(samples))

    plt.figure(figsize=(10, 4))
    plt.plot(time_axis, samples)
    plt.title("Your Voice Waveform")
    plt.xlabel("Time (seconds)")
    plt.ylabel("Amplitude")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

def main():
    print("=" * 40)
    print("🎙️ HELLO AI, CAN YOU HEAR ME?")
    print("=" * 40)

    webm_file = record_audio_colab(seconds=5)
    wav_file = convert_webm_to_wav(webm_file, "recording.wav")

    display(Audio(wav_file))

    audio_data, rate, width = read_wav_bytes(wav_file)
    save_audio_info(wav_file)
    transcribe(audio_data, rate, width)
    plot_waveform(audio_data, rate)

main()

If you are on macOS and encounter issues with `PyAudio`, you might need to install `portaudio` first using Homebrew:

`brew install portaudio`